# Compare Original vs Rendered Vid2Room Frames

Creates a side-by-side MP4 video comparing the original video frames with the OmniGibson-rendered frames for a given scene.

**Prerequisites:** Run `render_vid2room_scenes.py` first to generate rendered frames.

**Inputs:**
- `room_dir`: Path to the vid2room room (original images at `room_dir/images/`)
- `render_dir`: Path to the rendered output (e.g. `{dataset}/vid2room/renders/{scene_id}`). If None, inferred from `room_dir`.
- `output_mp4`: Where to save the comparison video. If None, uses `./{scene_id}_original_vs_rendered.mp4`.

**Usage:** Set `room_dir` in the config cell (or uncomment the pick-by-index cell to choose from `interesting_scenes.json`), then run all cells.

In [1]:
import pathlib
import json
import numpy as np
from PIL import Image
import cv2
from tqdm.auto import tqdm

In [2]:
# Optional: pick a scene by index from interesting_scenes.json (uncomment to use)
# SCENE_INDEX = 4  # vid_--dubp2RBuc/rooms/living_room_0
# scene_list = json.loads(pathlib.Path("/cvgl2/u/cgokmen/BEHAVIOR-1K/slurm/interesting_scenes.json").read_text())
# room_dir = pathlib.Path(scene_list[SCENE_INDEX])

In [3]:
# --- Configuration: set these for your scene ---
room_dir = pathlib.Path("/vision/group/vid2room/raw/RealEstate10K/vid_-bI3OU811Vk/rooms/bedroom_1")

# Rendered output directory (default: <dataset>/vid2room/renders/<scene_id>)
dataset_root = pathlib.Path("/cvgl2/u/cgokmen/BEHAVIOR-1K/datasets")
scene_id = None  # If None, inferred from room_dir via get_scene_id()
render_dir = None  # If None, uses dataset_root/vid2room/renders/<scene_id>

output_mp4 = None  # If None, uses ./original_vs_rendered.mp4
fps = 10

In [4]:
def get_scene_id(room_dir):
    """Generate scene ID from room path: vid_XXXXX_room_type_N"""
    room_name = pathlib.Path(room_dir).name
    video_id = pathlib.Path(room_dir).parent.parent.name
    assert video_id.startswith("vid_"), f"Video ID {video_id} does not start with 'vid_'"
    return f"{video_id}_{room_name}"

In [5]:
# Infer scene_id, render_dir, output_mp4 if not set
if scene_id is None:
    scene_id = get_scene_id(room_dir)
    print(f"Inferred scene_id: {scene_id}")
if render_dir is None:
    render_dir = dataset_root / "vid2room" / "renders" / scene_id
if output_mp4 is None:
    output_mp4 = pathlib.Path(".") / f"{scene_id}_original_vs_rendered.mp4"

Inferred scene_id: vid_-bI3OU811Vk_bedroom_1


In [6]:
# Load metadata from render output
metadata_path = render_dir / "metadata.json"
if not metadata_path.exists():
    raise FileNotFoundError(f"No metadata at {metadata_path}. Run render_vid2room_scenes first.")

metadata = json.loads(metadata_path.read_text())
filenames = metadata["filenames"]
num_frames = metadata["num_frames"]
render_w = metadata["render_width"]
render_h = metadata["render_height"]

print(f"Scene: {scene_id}")
print(f"Frames: {num_frames}")
print(f"Render resolution: {render_w}x{render_h}")

Scene: vid_-bI3OU811Vk_bedroom_1
Frames: 56
Render resolution: 1920x1080


In [7]:
# Verify paths exist
rgb_dir = render_dir / "rgb"
images_dir = room_dir / "images"

if not rgb_dir.exists():
    raise FileNotFoundError(f"No rgb dir at {rgb_dir}")
if not images_dir.exists():
    raise FileNotFoundError(f"No images dir at {images_dir}")

# Check first frame
first_orig = images_dir / filenames[0]
first_rend = rgb_dir / "00000.png"
if not first_orig.exists():
    raise FileNotFoundError(f"Original frame not found: {first_orig}")
if not first_rend.exists():
    raise FileNotFoundError(f"Rendered frame not found: {first_rend}")

print("Paths OK")

Paths OK


In [8]:
# Determine output size: side-by-side, both at same height for fair comparison
# Resize to common height if needed
target_h = render_h
target_w_per_side = render_w  # each side gets render resolution width
out_w = target_w_per_side * 2
out_h = target_h

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(output_mp4), fourcc, fps, (out_w, out_h))

In [9]:
for i in tqdm(range(num_frames), desc="Writing frames"):
    # Load original
    orig_path = images_dir / filenames[i]
    orig = np.array(Image.open(orig_path).convert("RGB"))
    if orig.shape[:2] != (target_h, target_w_per_side):
        orig = cv2.resize(orig, (target_w_per_side, target_h), interpolation=cv2.INTER_LINEAR)

    # Load rendered (RGBA from OmniGibson)
    rend_path = rgb_dir / f"{i:05d}.png"
    rend = np.array(Image.open(rend_path).convert("RGB"))
    if rend.shape[:2] != (target_h, target_w_per_side):
        rend = cv2.resize(rend, (target_w_per_side, target_h), interpolation=cv2.INTER_LINEAR)

    # Side-by-side: original | rendered
    frame = np.hstack([orig, rend])

    # Add labels
    cv2.putText(frame, "Original", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    cv2.putText(frame, "Rendered", (target_w_per_side + 10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

Writing frames:   0%|          | 0/56 [00:00<?, ?it/s]

In [10]:
writer.release()
print(f"Saved: {output_mp4}")

Saved: vid_-bI3OU811Vk_bedroom_1_original_vs_rendered.mp4
